# Duka Electronics — H1 2026 Sales Exploration

## Executive Summary
- **Nairobi** is the top branch with KES 22,925,800 (32.4% of the KES 70,678,600 total revenue across 1,000 orders).
- **Laptop** is the top product by revenue at KES 18,972,000, while **Headphones** leads on units with 608 sold.
- Weekend orders are 27.9% of orders (279), averaging KES 65,970 per order vs KES 72,501 on weekdays.
- **March** was the best month (KES 13,930,900) and **May** the worst (KES 9,832,700), a 41.7% gap.
- 71 Premium online orders from Nairobi and Mombasa generated KES 8,784,000, i.e. 12.4% of revenue.

## Task 1 · Load & first-five-minutes inspection

In [1]:
import pandas as pd, numpy as np
pd.options.display.float_format = "{:,.2f}".format
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("shape:", df.shape); print(df.dtypes); print("\nmissing values:\n", df.isna().sum())
print("\ndate range:", df.order_date.min().date(), "to", df.order_date.max().date())
for c in ["branch","product","channel"]: print("\n", df[c].value_counts().to_dict())
print("duplicate order_id:", df.order_id.duplicated().sum())
df.describe()

shape: (1000, 7)
order_id                 str
order_date    datetime64[us]
branch                   str
product                  str
channel                  str
units                  int64
unit_price             int64
dtype: object

missing values:
 order_id      0
order_date    0
branch        0
product       0
channel       0
units         0
unit_price    0
dtype: int64

date range: 2026-01-01 to 2026-06-30

 {'Nairobi': 307, 'Mombasa': 208, 'Nakuru': 166, 'Kisumu': 161, 'Eldoret': 158}

 {'Headphones': 202, 'Smartphone': 129, 'Tablet': 129, 'Router': 125, 'Smartwatch': 122, 'Printer': 105, 'Laptop': 98, 'Television': 90}

 {'In-store': 577, 'Online': 423}
duplicate order_id: 0


,order_date,units,unit_price
count,1000,"1,000.00","1,000.00"
mean,2026-03-30 20:44:09.600000,3.03,"23,020.00"
min,2026-01-01 00:00:00,1.00,"6,500.00"
25%,2026-02-15 00:00:00,2.00,"7,800.00"
50%,2026-03-29 12:00:00,3.00,"18,500.00"
75%,2026-05-15 00:00:00,4.00,"28,500.00"
max,2026-06-30 00:00:00,5.00,"62,000.00"
std,NaN,1.40,"17,023.57"


**Inspection notes.** 1,000 rows × 7 columns. `order_date` parsed to datetime (range 2026-01-01 → 2026-06-30); (0 nulls, 0 duplicates). Categories: 5 branches, 8 products, channels In-store, Online.
> **Data-source note:** `duka_sales.csv` in this build is a stand-in (`standin_duka_generator.py`). Replace it with the course-generated file and run `python build_duka.py` to refresh every number below.

## Task 2 · Derived columns

In [2]:
df["revenue"] = df["units"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")
df[["order_id","revenue","month","weekday","is_weekend","price_band"]].head()

,order_id,revenue,month,weekday,is_weekend,price_band
0,D0001,225000,June,Thursday,False,Premium
1,D0002,13000,February,Monday,False,Value
2,D0003,13000,January,Monday,False,Value
3,D0004,39000,April,Sunday,True,Value
4,D0005,39000,March,Sunday,True,Value


`revenue = units × unit_price`; `price_band` uses `unit_price ≥ KES 28,000` → Premium.

## Task 3 · Branch ranking: revenue vs orders

In [3]:
b = df.groupby("branch").agg(total_revenue=("revenue","sum"), orders=("order_id","count"))
b["avg_order_value"] = b.total_revenue/b.orders
b["rank_revenue"] = b.total_revenue.rank(ascending=False).astype(int)
b["rank_orders"] = b.orders.rank(ascending=False, method="min").astype(int)
b.sort_values("total_revenue", ascending=False)

,total_revenue,orders,avg_order_value,rank_revenue,rank_orders
branch,,,,,
Nairobi,22925800,307,"74,676.87",1,1
Mombasa,13252400,208,"63,713.46",2,2
Kisumu,12464900,161,"77,421.74",3,4
Nakuru,11404800,166,"68,703.61",4,3
Eldoret,10630700,158,"67,282.91",5,5


**Answer.** By revenue: Nairobi > Mombasa > Kisumu > Nakuru > Eldoret. By orders: Nairobi > Mombasa > Nakuru > Kisumu > Eldoret. The rankings differ. **Nairobi** leads on revenue but **Nairobi** leads on orders: revenue = orders × average order value, so a branch with fewer orders can out-earn one with more if it sells higher-priced items (e.g. Nairobi AOV KES 74,677 vs Nairobi KES 74,677).

## Task 4 · Top three products

In [4]:
p = df.groupby("product").agg(revenue=("revenue","sum"), units=("units","sum"))
print("By revenue:"); display(p.nlargest(3,"revenue"))
print("By units:");   display(p.nlargest(3,"units"))

By revenue:


,revenue,units
product,,
Laptop,18972000,306
Television,12915000,287
Smartphone,10915500,383


By units:


,revenue,units
product,,
Headphones,3952000,608
Tablet,9336000,389
Smartphone,10915500,383


**Answer.** Top by revenue: Laptop, Television, Smartphone. Top by units: Headphones, Tablet, Smartphone. The lists differ because revenue rewards price × volume: expensive items earn heavily from fewer units, while cheap accessories sell many units but contribute little revenue.

## Task 5 · Weekend vs weekday

In [5]:
w = df.groupby("is_weekend").agg(orders=("order_id","count"), total_revenue=("revenue","sum"), avg_order_revenue=("revenue","mean"))
w.index = w.index.map({True:"Weekend", False:"Weekday"}); w

,orders,total_revenue,avg_order_revenue
is_weekend,,,
Weekday,721,52273000,"72,500.69"
Weekend,279,18405600,"65,969.89"


**Answer.** Weekdays: 721 orders, KES 52,273,000, avg KES 72,501. Weekends: 279 orders, KES 18,405,600, avg KES 65,970. Weekends cover only 2 of 7 days, so compare the average order value, not totals: weekend orders are smaller on average (-9.0%).

## Task 6 · Best and worst month

In [6]:
m = df.groupby(df.order_date.dt.month).revenue.sum()
m.index = [pd.Timestamp(2026,i,1).month_name() for i in m.index]
best, worst = m.idxmax(), m.idxmin()
print(m); print(f"best={best} worst={worst} difference={(m[best]-m[worst])/m[worst]*100:.1f}% (best vs worst)")

January     11908500
February    11044600
March       13930900
April       11298500
May          9832700
June        12663400
Name: revenue, dtype: int64
best=March worst=May difference=41.7% (best vs worst)


**Answer.** Best: **March** (KES 13,930,900); worst: **May** (KES 9,832,700). Best is **41.7%** higher than worst, i.e. worst is 29.4% below best.

## Task 7 · Premium online orders, Nairobi & Mombasa

In [7]:
sel = df[(df.price_band=="Premium") & (df.channel=="Online") & (df.branch.isin(["Nairobi","Mombasa"]))]
print(len(sel), "orders | revenue KES", f"{sel.revenue.sum():,.0f}")
sel.to_csv("outputs/premium_online_coast_capital.csv", index=False)

71 orders | revenue KES 8,784,000


**Answer.** **71** orders, total revenue **KES 8,784,000**. Exported to `outputs/premium_online_coast_capital.csv` (`index=False`).

## Task 8 · Excel reconciliation (`excel/duka_branch_summary.xlsx`)

In [8]:
xl = pd.read_csv("excel/sumifs_recalc_check.csv").set_index("branch")      # SUMIFS/COUNTIFS values recalculated from the workbook
cmp = b[["total_revenue","orders"]].join(xl)
cmp["match"] = np.isclose(cmp.total_revenue, cmp.sumifs_revenue) & (cmp.orders == cmp.countifs_orders)
raw = pd.read_csv("excel/pivot_recalc_check.csv", header=None)         # values of the real PivotTable in the workbook
hdr_i = raw.index[raw.iloc[:,0].astype(str).str.contains("branch", case=False)][-1]
pv = raw.iloc[hdr_i+1:].copy(); pv.columns = ["branch"]+list(raw.iloc[hdr_i,1:]); pv = pv.set_index("branch")
pd_pivot = df.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
ok = np.allclose(pv.loc[pd_pivot.index, pd_pivot.columns].astype(float), pd_pivot)
tot = pv.loc[pv.index.str.contains("Total", case=False)].iloc[0].astype(float).iloc[-1]
print("SUMIFS = pandas for every branch:", cmp.match.all(), "| grand total:", f"{xl.sumifs_revenue.sum():,.0f} vs {df.revenue.sum():,.0f}")
print("PivotTable branch×channel = pandas pivot:", ok, "| pivot grand total:", f"{tot:,.0f}")
cmp

SUMIFS = pandas for every branch: True | grand total: 70,678,600 vs 70,678,600
PivotTable branch×channel = pandas pivot: True | pivot grand total: 70,678,600


,total_revenue,orders,sumifs_revenue,countifs_orders,match
branch,,,,,
Eldoret,10630700,158,10630700,158,True
Kisumu,12464900,161,12464900,161,True
Mombasa,13252400,208,13252400,208,True
Nairobi,22925800,307,22925800,307,True
Nakuru,11404800,166,11404800,166,True


**Answer.** The workbook has `Branch_Summary` (SUMIFS revenue + COUNTIFS orders + XLOOKUP region from the `Regions` sheet) and `Pivot_Branch_Channel` (a real PivotTable: branch rows × channel columns, sum of revenue). The check above shows every total equals pandas.